# Stage 1：p=5 Student-t，M3 一步修正 + Wald 适配版

本 Notebook 为独立源码展开版，不导入 benchmark。默认 smoke：n=200、R=3、K=5，2 DGP、Gaussian/t(20)/t(2.1)，72 条方法终态记录。

**根据用户最新决定，M3 取消置信集合反演，也取消原来的反复重拟合求根。**
M3 在 M1 初值处只拟合一次变换结果辅助模型（5 折），冻结平滑方向，解析计算一步修正和配套 plug-in Wald 区间。
方法标签为 Dukes-inspired one-step/frozen-path Wald adaptation；不是 Dukes 原文检验反演程序的严格复现，原文的双重稳健推断保证不能直接移植。

用户确认：Huber 方差分母 n−1；M3 完整 score 方差分母 n。
Codex 仅修改文件及静态检查，不代运行模型、模拟、smoke/full。旧运行数据、备份和生成脚本按用户要求清理；来源路径与哈希仍嵌入本文档。


## 本次改动与耗时依据

原 smoke 的 18 个 M3 任务中，10 个在点估计求根阶段失败，尚未反演就耗时约 42～65 秒，说明仅更换区间不足以解决开销。
因此采用用户授权的简化程序：1 个初值、5 次变换结果模型拟合、10 次一维带宽选择；区间计算不训练模型，不搜索端点。
旧成功任务未分段计时，无法准确拆出其点估计耗时。新记录分别保存 point_elapsed_seconds 和 ci_elapsed_seconds。

max_features=1.0 的类型修复和全失败汇总处理继续保留。版本、方法签名和 run_id 均已更新。


## 1. 导入依赖

这里仅导入科学计算库，不导入项目自定义模块。依赖需要由用户在当前 Jupyter 内核对应环境安装。
如果某个包不存在，错误会列出内核 Python 路径和安装命令；代码不会自动安装。


In [ ]:
import ast
import hashlib
import importlib.metadata as metadata
import inspect
import json
import math
import os
from pathlib import Path
import platform
import sys
import textwrap
import time
import traceback
import warnings

try:
    import numpy as np
    import pandas as pd
    from scipy.optimize import brentq, minimize_scalar
    from scipy.stats import norm
    from sklearn.ensemble import HistGradientBoostingRegressor
    from sklearn.model_selection import KFold
    from threadpoolctl import threadpool_limits
    from IPython.display import display
except ModuleNotFoundError as exc:
    raise RuntimeError(
        f"当前内核缺少依赖 {exc.name}。内核 Python：{sys.executable}\n"
        "请由用户在此内核环境安装 requirements.txt，然后重启内核。"
    ) from exc


## 2. 集中配置

只修改 MODE 即可切换 smoke/full（standard 是 full 别名）。full 入口检查本版本通过的 smoke 报告、依赖版本、方法签名和完整正式网格；不自动启动 full。
STOP_AFTER_REPS=2 可供用户检查中断与恢复，恢复时改回 None。所有模式都由用户执行。
项目路径优先使用 Notebook 工作目录，再读取系统实际桌面位置；可使用 PROJECT_ROOT_OVERRIDE 手动指定。


In [ ]:
PROJECT_ROOT_OVERRIDE = None
MODE = "full"
MODE = "full" if MODE == "standard" else MODE
if MODE not in ("smoke", "full"):
    raise ValueError('MODE 必须是 smoke、full 或 standard')
STAGE_ID = "stage_01"
RESUME = True
RETRY_FAILED = False
N_JOBS = 4
ALPHA = 0.05
BASE_SEED = 20261103
STOP_AFTER_REPS = None  # 开发中断检查可设 2，恢复时改回 None。

def locate_project_root(override=None):
    if override is not None:
        return Path(override).expanduser().resolve()
    current = Path.cwd().resolve()
    if (current / "DML_p5_student_t.ipynb").is_file():
        return current
    if os.name == "nt":
        import winreg
        registry_path = r"Software\Microsoft\Windows\CurrentVersion\Explorer\User Shell Folders"
        with winreg.OpenKey(winreg.HKEY_CURRENT_USER, registry_path) as registry:
            desktop_value, _ = winreg.QueryValueEx(registry, "Desktop")
        desktop = Path(os.path.expandvars(desktop_value))
        return (desktop / "DML_HeavyTail_Benchmark").resolve()
    # 在其他电脑/系统运行单独一份 Notebook 时，将结果放在当前工作目录下。
    return (current / "DML_HeavyTail_Benchmark").resolve()

PROJECT_ROOT = locate_project_root(PROJECT_ROOT_OVERRIDE)
print("项目目录：", PROJECT_ROOT)
print("当前内核：", sys.executable)
print("模式：", MODE, "；M3 为一步修正 + Wald 适配版本")


## 3. 冻结的实验设计、学习器和数值参数

HGBR 全参数取自旧 `legacy_config.json`，没有减少 smoke 的树数。
两份配置规模一起进入方法验收签名，当前 MODE 和输出路径不进入该签名。
原始来源清单嵌入本 Notebook，便于复制文件后继续审计；原始快照是否存在会另行核验。


In [ ]:
VERSION = "stage1_v4_M3_one_step_Wald"
METHODS = ('M1_STANDARD_DML', 'M2_WANG2024_RDML', 'M3_DUKES', 'M4_HUBER')
VARIANTS = ('HGBR_mean_OLS', 'Wang-style_HGBR_5fold',
            'Dukes-inspired_one_step_frozen_path_Wald_v1', 'symmetric_fixed_tau_no_intercept')
DGP_NAMES = ('DGP1_linear', 'DGP2_nonlinear')
T_GRID = (20., 10., 5., 3., 2.6, 2.3, 2.1)
DESIGNS = {'smoke': {'n': 200, 'R': 3, 'errors': [0, 1, 7]},
           'full': {'n': 1000, 'R': 1000, 'errors': list(range(8))}}
SOLVER = {'min_denominator': 1e-10, 'm3_condition_limit': 1e8,
          'score_tol': 1e-8, 'kappa': 1.345, 'm3_steps': 1,
          'm3_path': 'm(theta)=m(anchor)-(theta-anchor)*g; G,M,alpha fixed; beta reprojected analytically',
          'm3_se': 'frozen_path_centered_sandwich_ddof0'}

DEFAULT_KERNEL_CONFIG = {'bwtype': 'fixed',
 'constant_index': 'mean_response',
 'cv': 'leave_one_out_squared_error',
 'final_include_self': True,
 'grid_size': 41,
 'implementation': 'python_gaussian_nw_loocv_v1',
 'kernel': 'gaussian_order2',
 'log_bandwidth_xatol': 1e-06,
 'lower_multiplier': 0.05,
 'refine': True,
 'refine_maxiter': 80,
 'regtype': 'local_constant',
 'scale': 'sample_sd_times_fold_n_power_minus_0.2',
 'upper_multiplier': 20.0}

HGBR_PARAMS = {'categorical_features': 'from_dtype',
 'early_stopping': 'auto',
 'interaction_cst': None,
 'l2_regularization': 1,
 'learning_rate': 0.05,
 'loss': 'squared_error',
 'max_bins': 255,
 'max_depth': None,
 'max_features': 1.0,
 'max_iter': 100,
 'max_leaf_nodes': 15,
 'min_samples_leaf': 20,
 'monotonic_cst': None,
 'n_iter_no_change': 10,
 'quantile': None,
 'random_state': 0,
 'scoring': 'loss',
 'tol': 1e-07,
 'validation_fraction': 0.1,
 'verbose': 0,
 'warm_start': False}

SOURCE_INVENTORY = [{'id': 'requirements_readme',
  'path': 'C:/Users/Lenovo/Documents/ChatGPT/毕设/README.md',
  'sha256': 'd9556be991eb8021fc4b8466e35bb4e82665c9ea10886329e490418a72d1a746',
  'snapshot': 'sources/requirements_readme.md'},
 {'id': 'student_original',
  'path': 'C:/Users/Lenovo/Desktop/DML论文/DML_p5_student_t_heavytail.ipynb',
  'sha256': '7ae8901f8bd97e8cd6420331c6b0b31910116e56c80429ac56cc33dcbb5b0a32',
  'snapshot': 'sources/student_original.ipynb'},
 {'id': 'dukes_original',
  'path': 'C:/Users/Lenovo/Desktop/毕设/DML_Dukes_n1000_R1000.ipynb',
  'sha256': 'e80fbf528039c1c554b63b4a9cfbf96af5cab70f85f5da491f3934049c8c8394',
  'snapshot': 'sources/dukes_original.ipynb'},
 {'id': 'huber_original',
  'path': 'C:/Users/Lenovo/Desktop/DML_Stage2_Robust_M_Pilot/stage2_estimators.py',
  'sha256': '1806a10181cea18256bfde62a01222d2cc5bb5e349d685bcfe75fb5fc1422958',
  'snapshot': 'sources/huber_original.py'},
 {'id': 'huber_loss',
  'path': 'C:/Users/Lenovo/Desktop/DML_Stage2_Robust_M_Pilot/stage2_losses.py',
  'sha256': '928fb4147ec9b9e411503f6af05e6403162269ea69fc4f2bc24fa61dc218fa29',
  'snapshot': 'sources/huber_loss.py'},
 {'id': 'legacy_config',
  'path': 'C:/Users/Lenovo/Desktop/DML_Stage2_Robust_M_Pilot/legacy_config.json',
  'sha256': 'b5e60a5f198988cdc7ddce096537049b4f7131f9e7b0021fe8a42afdf46d8f83',
  'snapshot': 'sources/legacy_config.json'},
 {'id': 'author_R',
  'path': 'C:/Users/Lenovo/Desktop/DML_Dukes_full_results/R_cross_validation/downloads/dri_functions.r',
  'sha256': 'fa32ffae449144fb0bf85c894f2cd9271bc5f492657f289db324da8db4b9bd71',
  'snapshot': 'sources/author_R.r'},
 {'id': 'author_comparison',
  'path': 'C:/Users/Lenovo/Desktop/DML_Dukes_full_results/R_cross_validation/cross_validation_report.md',
  'sha256': 'fa2731456c54ff46401987a40262c0b078f76d7dc788d8ca841a8d8bb13cc3d1',
  'snapshot': 'sources/author_comparison.md'},
 {'id': 'paper_text',
  'path': 'C:/Users/Lenovo/Documents/ChatGPT/毕设/tmp/pdfs/dukes_2024.txt',
  'sha256': '268132feab2fc2e70e206ba8e9cb1e5a77b322f9a139ba33d6ee94d21efcc647',
  'snapshot': 'sources/paper_text.txt'}]


## 4. 签名、原子写入和运行配置

源码签名取自当前内核里已定义函数的 AST，不依赖 `__file__`，也不读取外部 benchmark.py。
当前 Notebook 的执行输出、执行计数、MODE 选择和纯调度开关不会改变方法签名。
本单元只定义函数；实际配置在所有函数定义完成后构造。


In [ ]:
def sha(value):
    return hashlib.sha256(value).hexdigest()


def digest(obj):
    return sha(json.dumps(obj, sort_keys=True, ensure_ascii=False).encode())


def atomic_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(obj, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
    os.replace(temp, path)


def versions():
    return dict(python=platform.python_version(), **{
        name: metadata.version(name) for name in
        ('numpy', 'scipy', 'scikit-learn', 'pandas', 'matplotlib', 'threadpoolctl')})


def implementation_fingerprint():
    sources = {}
    for name in IMPLEMENTATION_NAMES:
        obj = globals()[name]
        members = {name: obj} if inspect.isfunction(obj) else {
            name + "." + key: value for key, value in vars(obj).items()
            if inspect.isfunction(value)
        }
        for key, function in members.items():
            try:
                source = textwrap.dedent(inspect.getsource(function))
            except (OSError, TypeError) as exc:
                raise RuntimeError(
                    f"无法取得 {key} 源码签名。请使用 Jupyter 新内核，从头执行所有定义单元。"
                ) from exc
            sources[key] = ast.dump(ast.parse(source), include_attributes=False)
    return digest(sources)

def configuration(project_root, mode="smoke", base_seed=20261103, alpha=.05):
    mode = "full" if mode == "standard" else mode
    if mode not in DESIGNS:
        raise ValueError("无效模式")
    project_root = Path(project_root).resolve()
    code_hash = implementation_fingerprint()
    spec = dict(
        version=VERSION, code_hash=code_hash, designs=DESIGNS, solvers=SOLVER.copy(),
        learner=HGBR_PARAMS.copy(), kernel=DEFAULT_KERNEL_CONFIG.copy(),
        alpha=alpha, base_seed=base_seed, seed_rule="SeedSequence_v1", p=5, K=5,
        huber_ddof=1, dukes_ddof=0, wald_critical="1.96 at alpha .05",
        sources=digest(SOURCE_INVENTORY)
    )
    method_signature = digest(spec)
    actual = dict(mode=mode, **DESIGNS[mode], p=5, K=5, base_seed=base_seed,
                  alpha=alpha, dependencies=versions(), method_signature=method_signature)
    signature = digest(actual)
    return dict(actual, spec=spec, config_hash=signature, code_hash=code_hash,
                stage_id="stage_01", project_root=str(project_root),
                run_id=mode + "_" + signature[:16])


## 5. 原始 DGP 和 Student-t 标准化

PLR：Y=D+g0(X)+ε，D=m0(X)+v；v~N(0,1)，X 的协方差是 0.5^|j−k|。
下面三函数直接复制原 Student-t Notebook 的 cell index 4（未复制演示运行代码）。
Student-t 仅使用总体矩标准化，不按样本方差重缩放，不裁剪极端样本。


In [ ]:
def ar1_covariance(p: int, rho: float = 0.5) -> np.ndarray:
    """构造 Sigma[j,k] = rho**|j-k| 的 AR(1) 型协方差矩阵。"""
    index = np.arange(p)
    return rho ** np.abs(index[:, None] - index[None, :])


def nuisance_functions(X: np.ndarray, dgp_name: str):
    """返回指定 DGP 的 m0(X) 与 g0(X)。"""
    x1, x2, x3, x4, x5 = X.T

    if dgp_name == "DGP1_linear":
        m0 = 0.80 * x1 - 0.50 * x2 + 0.30 * x3 + 0.20 * x4 - 0.10 * x5
        g0 = 1.00 * x1 + 0.50 * x2 - 0.50 * x3 + 0.25 * x4 + 0.10 * x5
    elif dgp_name == "DGP2_nonlinear":
        # x2**2 - 1 做中心化，因为标准正态变量满足 E[X^2]=1
        m0 = (
            0.60 * np.sin(x1)
            + 0.35 * (x2**2 - 1.0)
            + 0.25 * x3 * x4
            - 0.20 * np.tanh(x5)
        )
        g0 = (
            1.20 * np.sin(x1)
            + 0.70 * (x2**2 - 1.0)
            + 0.50 * x3 * x4
            + 0.30 * np.tanh(x5)
        )
    else:
        raise ValueError(f"未知 DGP: {dgp_name}")
    return m0, g0


def standardized_student_t(rng: np.random.Generator, df: float, size: int) -> np.ndarray:
    """生成均值 0、方差 1 的 Student-t 误差；要求 df > 2。"""
    if df <= 2:
        raise ValueError("Student-t 自由度必须大于 2，才能标准化为单位方差。")
    return rng.standard_t(df, size=size) * np.sqrt((df - 2.0) / df)


## 6. 独立子流、共享 folds 与 OOF 学习

随机键含 base seed、模式、p、DGP、n、error、replication、purpose、fold、learner。
Gaussian 不含阶段名或 Notebook 名，可供后续相同 p 的基准重跑核验。
X/v/ε/folds 分别生成；四方法共享观测数据和外层 folds。初始 nuisance 只在训练折拟合。


In [ ]:
def seed(config, dgp, error, rep, purpose, fold=0, learner=0):
    key = [config['base_seed'], {'smoke': 1, 'full': 2}[config['mode']],
           config['p'], dgp, config['n'], error, rep, purpose, fold, learner]
    return int(np.random.SeedSequence(key).generate_state(1)[0])


def generate_observed(config, dgp, error, rep):
    n = config['n']
    seeds = {name: seed(config, dgp, error, rep, purpose)
             for name, purpose in [('X', 1), ('v', 2), ('epsilon', 3), ('folds', 4)]}
    X = np.random.default_rng(seeds['X']).multivariate_normal(
        np.zeros(5), ar1_covariance(5), n)
    m, g = nuisance_functions(X, DGP_NAMES[dgp])
    v = np.random.default_rng(seeds['v']).normal(size=n)
    rng = np.random.default_rng(seeds['epsilon'])
    epsilon = rng.normal(size=n) if error == 0 else standardized_student_t(rng, T_GRID[error-1], n)
    D = m + v
    Y = D + g + epsilon  # theta_true=1 is confined to generation/evaluation.
    assert X.shape == (n, 5) and np.isfinite(np.r_[X.ravel(), D, Y]).all()
    folds = list(KFold(5, shuffle=True, random_state=seeds['folds']).split(X))
    validate_folds(folds, n)
    assert n % 5 == 0
    fold_ids = np.empty(n, dtype=np.int64)
    for k, (_, test) in enumerate(folds):
        fold_ids[test] = k
    audit = dict(seeds=seeds, data_hash=sha(X.tobytes()+D.tobytes()+Y.tobytes()),
                 fold_hash=sha(fold_ids.tobytes()),
                 epsilon_sample_variance=float(np.var(epsilon, ddof=1)))
    return (X, D, Y), folds, fold_ids, audit


def learner(config, random_state, median=False):
    params = config['spec']['learner'].copy()
    # sklearn 1.7 requires a float here; JSON conversions can erase .0.
    params['max_features'] = float(params['max_features'])
    params.update(random_state=random_state, loss='quantile' if median else 'squared_error',
                  quantile=.5 if median else None)
    return HistGradientBoostingRegressor(**params)


def crossfit(observed, folds, config, dgp, error, rep, median=False, candidate=None):
    X, D, Y = observed
    predictions = np.empty((2, len(D)))
    seen = np.zeros(len(D), dtype=int)
    for k, (train, test) in enumerate(folds):
        assert not np.intersect1d(train, test).size
        for j, target in enumerate((D, Y if candidate is None else Y-candidate*D)):
            s = seed(config, dgp, error, rep, 5 if candidate is None else 6,
                     k, j+2*int(median))
            model = learner(config, s, median)
            predictions[j, test] = model.fit(X[train], target[train]).predict(X[test])
        seen[test] += 1
    assert np.all(seen == 1) and np.isfinite(predictions).all()
    return predictions


## 7. M1：Standard DML

u=Y−ℓhat，w=D−mhat；θ=sum(wu)/sum(w²)，无截距。
SE=SD[w(u−θw)/mean(w²),ddof=1]/sqrt(n)，Wald 使用 1.96（ALPHA=.05）。


In [ ]:
def ols_point(w, u):
    J = float(np.mean(w*w))
    if J <= SOLVER['min_denominator']:
        raise ArithmeticError('degenerate_OLS_denominator')
    return float(np.mean(w*u)/J)


def m1(w, u):
    t = ols_point(w, u)
    q = w*(u-t*w)
    J = float(np.mean(w*w))
    se = float(np.std(q/J, ddof=1)/np.sqrt(len(w)))
    assert abs(float(w@(u-t*w))) < 1e-7*max(1., float(abs(w*u).sum()))
    return dict(theta_hat=t, se=se, se_type='IF_ddof1', ci_type='Wald',
                diagnostics=dict(J=J, score_error=float(abs(q.mean()))))


## 8. M2：Wang-style RDML（HGBR 适配）

条件中位数 HGBR；二阶段 LAD，通过 |w| 加权的 u/w 中位数求解，并列取较小值。
检查完整样本的 LAD 次梯度；推断沿用原 MAD 带宽的 kernel plug-in sandwich。
此实现不是 Wang 原文的中位数森林/XGBoost 软件严格复现。


In [ ]:
def m2(w, u):
    keep = abs(w) > 1e-10  # Original pilot threshold, retained explicitly.
    if not np.any(keep):
        raise ArithmeticError('degenerate_LAD_design')
    values, weights = u[keep]/w[keep], abs(w[keep])
    order = np.argsort(values, kind='stable')
    t = float(values[order[np.searchsorted(np.cumsum(weights[order]), weights.sum()/2, side='left')]])
    r = u-t*w
    # Independent LAD subgradient check includes observations discarded for ratio construction.
    zero = abs(r) <= 1e-9*np.maximum(1., abs(u)+abs(t*w))
    signed = float(np.sum(w[~zero]*np.sign(r[~zero])))
    radius = float(abs(w[zero]).sum())
    if abs(signed) > radius + 1e-7:
        raise ArithmeticError('LAD_subgradient_not_zero')
    scale = max(float(np.median(abs(r-np.median(r)))/.67448975), 1e-6)
    h = max(1.06*scale*len(r)**(-.2), 1e-6)
    A = float(np.mean(w*w*norm.pdf(r/h)/h))
    out = dict(theta_hat=t, se_type='LAD_kernel_plugin', ci_type='LAD_plugin_Wald',
               diagnostics=dict(A=A, bandwidth=h, scale=scale, signed=signed,
                                subgradient_radius=radius, tie_rule='lower_weighted_median'))
    out['se'] = float(np.sqrt(.25*np.mean(w*w)/(len(w)*A*A))) if A > 1e-12 else None
    return out


## 9. M4：二阶段固定阈值对称 Huber

复用 M1 的同一份均值 OOF 预测。OLS 残差 MAD×1.4826 得 s；τ=1.345s。
固定 τ 求 mean[w clip(u−θw,−τ,τ)]=0，不加截距、惩罚或 Oracle 修偏。
SE=SD(q/A,ddof=1)/sqrt(n)，A=mean[w²1{|r|<τ}]；阈值边界取导数 0。
用户确认的 n−1 与原 pilot 的 n 差异已记录。


In [ ]:
def m4(w, u):
    initial = ols_point(w, u)
    r0 = u-initial*w
    s = float(1.4826*np.median(abs(r0-np.median(r0))))
    if not np.isfinite(s) or s <= 1e-12:
        raise ArithmeticError('degenerate_Huber_MAD')
    tau = SOLVER['kappa']*s
    def equation(t):
        return float(np.mean(w*np.clip(u-t*w, -tau, tau)))
    width = 1.
    for _ in range(40):
        if equation(initial-width) >= 0 >= equation(initial+width):
            break
        width *= 2
    else:
        raise ArithmeticError('Huber_bracket_failed')
    t = float(brentq(equation, initial-width, initial+width, xtol=1e-12, rtol=1e-14))
    r = u-t*w
    q = w*np.clip(r, -tau, tau)
    A = float(np.mean(w*w*(abs(r) < tau)))
    if abs(q.mean()) > 1e-9:
        raise ArithmeticError('Huber_score_residual')
    out = dict(theta_hat=t, se_type='Huber_plugin_ddof1_fixed_tau', ci_type='Huber_plugin_Wald',
               diagnostics=dict(scale=s, tau=tau, A=A, score_error=float(abs(q.mean())),
                                clipping=float(np.mean(abs(r)>tau)),
                                boundary_count=int(np.isclose(abs(r), tau, atol=1e-10, rtol=0).sum()),
                                objective=float(np.mean(np.where(abs(r)<=tau, .5*r*r, tau*abs(r)-.5*tau*tau)))))
    out['se'] = float(np.std(q/A, ddof=1)/np.sqrt(len(w))) if A > 1e-12 else None
    return out


## 10. M3 基础：Dukes 核平滑、校准和完整 score

下列定义复制原 Dukes Notebook cell index 8；未复制会自动启动运行的单元。
g≈E[D|X]，m(θ)≈E[Y−θD|X]。在评价折做一维 NW 平滑、无截距投影校准。
score 是 a b−G a−M b，两个减项必须保留在逐观测方差里。
Python 带宽选择与作者 R 的 npregbw 不等价，方法标签明确保留 adaptation。


In [ ]:
class NumericalFailure(ValueError):
    def __init__(self, failure_type, message, fold=None):
        super().__init__(message)
        self.failure_type = failure_type
        self.fold = fold


def _finite_vector(value, name):
    value = np.asarray(value, dtype=float)
    if value.ndim != 1 or not np.all(np.isfinite(value)):
        raise NumericalFailure("nonfinite_or_nonvector", f"{name} must be a finite vector")
    return value


def validate_folds(folds, n):
    """Validate full complementary train/test partitions with equal test sizes."""
    clean, seen, sizes = [], np.zeros(n, dtype=int), []
    for k, (train, test) in enumerate(folds):
        train, test = np.asarray(train, dtype=int), np.asarray(test, dtype=int)
        if (train.ndim != 1 or test.ndim != 1 or len(test) < 2 or
                len(np.unique(train)) != len(train) or len(np.unique(test)) != len(test) or
                np.any(train < 0) or np.any(train >= n) or np.any(test < 0) or np.any(test >= n) or
                np.intersect1d(train, test).size or
                not np.array_equal(np.sort(np.r_[train, test]), np.arange(n))):
            raise NumericalFailure("invalid_folds", "Each training set must be the test complement", k)
        seen[test] += 1
        sizes.append(len(test))
        clean.append((train, test))
    if not clean or np.any(seen != 1) or len(set(sizes)) != 1:
        raise NumericalFailure("invalid_folds", "Equal-sized test folds must partition observations")
    return clean


def _nw_from_distances(squared_distances, y, bandwidth, leave_one_out):
    if not np.isfinite(bandwidth) or bandwidth <= 0:
        raise NumericalFailure("invalid_bandwidth", "Bandwidth must be finite and positive")
    with np.errstate(over="ignore", under="ignore", invalid="ignore", divide="ignore"):
        logs = -0.5 * (squared_distances / bandwidth / bandwidth)
        if leave_one_out:
            np.fill_diagonal(logs, -np.inf)
        row_max = np.max(logs, axis=1)
        if not np.all(np.isfinite(row_max)):
            raise NumericalFailure("kernel_denominator", "No finite kernel weights")
        weights = np.exp(logs - row_max[:, None])
        predictions = (weights @ y) / weights.sum(axis=1)
    return _finite_vector(predictions, "NW predictions")


def nw_predict(x, y, bandwidth, leave_one_out=False):
    """NW fit evaluated at training positions (LOO only when explicitly requested)."""
    x, y = _finite_vector(x, "index"), _finite_vector(y, "response")
    if len(x) != len(y) or len(x) < 2:
        raise NumericalFailure("invalid_kernel_input", "Kernel vectors must have equal length >=2")
    return _nw_from_distances((x[:, None] - x[None, :]) ** 2, y, bandwidth, leave_one_out)


def select_nw(x, y, kernel_config=None):
    """Deterministic finite-grid LOOCV followed by local log-bandwidth refinement."""
    cfg = dict(DEFAULT_KERNEL_CONFIG)
    if kernel_config is not None:
        unknown = set(kernel_config) - set(cfg)
        if unknown:
            raise ValueError(f"Unknown kernel settings: {unknown}")
        cfg.update(kernel_config)
    for key in ("implementation", "kernel", "regtype", "bwtype", "cv", "scale", "constant_index", "final_include_self"):
        if cfg[key] != DEFAULT_KERNEL_CONFIG[key]:
            raise ValueError(f"Unsupported kernel choice {key}={cfg[key]}")
    x, y = _finite_vector(x, "index"), _finite_vector(y, "response")
    n = len(x)
    if n != len(y) or n < 2:
        raise NumericalFailure("invalid_kernel_input", "Kernel vectors must have equal length >=2")
    if np.all(x == x[0]):
        pred = np.full(n, np.mean(y))
        loo = (np.sum(y) - y) / (n - 1)
        return pred, {"bandwidth": None, "cv_loss": float(np.mean((y - loo)**2)),
                      "boundary": False, "boundary_side": "", "constant_index": True,
                      "bandwidth_low": None, "bandwidth_high": None, "optimizer_success": True}
    scale = float(np.std(x, ddof=1) * n ** (-0.2))
    lo, hi, count = float(cfg["lower_multiplier"]), float(cfg["upper_multiplier"]), int(cfg["grid_size"])
    if not (0 < lo < hi) or count < 3 or not np.isfinite(scale) or scale <= 0:
        raise NumericalFailure("invalid_bandwidth_grid", "Invalid bandwidth scale or grid")
    logs = np.linspace(np.log(scale * lo), np.log(scale * hi), count)
    distances = (x[:, None] - x[None, :]) ** 2

    def objective(log_h):
        try:
            pred = _nw_from_distances(distances, y, np.exp(log_h), True)
            loss = float(np.mean((y - pred) ** 2))
            return loss if np.isfinite(loss) else np.inf
        except NumericalFailure:
            return np.inf

    losses = np.array([objective(t) for t in logs])
    if not np.any(np.isfinite(losses)):
        raise NumericalFailure("no_valid_bandwidth", "All LOOCV candidates failed")
    best = int(np.argmin(losses))  # exact ties choose smaller h, deterministic
    best_log, best_loss, success = logs[best], losses[best], True
    if cfg["refine"] and 0 < best < count - 1:
        opt = minimize_scalar(objective, bounds=(logs[best-1], logs[best+1]), method="bounded",
                              options={"xatol": cfg["log_bandwidth_xatol"], "maxiter": cfg["refine_maxiter"]})
        success = bool(opt.success)
        if opt.success and opt.fun < best_loss:
            best_log, best_loss = float(opt.x), float(opt.fun)
    h = float(np.exp(best_log))
    return _nw_from_distances(distances, y, h, False), {
        "bandwidth": h, "cv_loss": float(best_loss), "boundary": best in (0, count-1),
        "boundary_side": "lower" if best == 0 else "upper" if best == count-1 else "",
        "constant_index": False, "bandwidth_low": float(np.exp(logs[0])),
        "bandwidth_high": float(np.exp(logs[-1])), "optimizer_success": success,
    }


def stable_projection(direction, target):
    """No-intercept 1D projection, scaling before squaring to avoid underflow."""
    q, target = _finite_vector(direction, "direction"), _finite_vector(target, "target")
    if len(q) != len(target):
        raise NumericalFailure("invalid_projection", "Projection lengths differ")
    scale = float(np.max(np.abs(q)))
    if scale == 0:
        return np.zeros_like(q), {"coefficient": 0.0, "direction_sum_squares": 0.0,
                                  "direction_norm": 0.0, "update_norm": 0.0, "zero_direction": True}
    normalized = q / scale
    denominator = float(normalized @ normalized)
    amount = float((normalized @ target) / denominator)
    update = normalized * amount
    with np.errstate(over="ignore", under="ignore"):
        coefficient = float(np.divide(amount, scale))
        raw_denominator = float(np.dot(q, q))
    if not np.isfinite(coefficient):
        raise NumericalFailure("nonfinite_coefficient", "Projection coefficient cannot be represented")
    _finite_vector(update, "update")
    return update, {"coefficient": coefficient, "direction_sum_squares": raw_denominator,
                    "direction_norm": float(scale * np.sqrt(denominator)),
                    "update_norm": float(np.linalg.norm(update)), "zero_direction": False}


def corrected_score(a, b, G, M):
    """Full pointwise score; correction terms must remain in its variance."""
    a, b, G, M = [_finite_vector(x, name) for x, name in ((a,"a"),(b,"b"),(G,"G"),(M,"M"))]
    if len({len(a), len(b), len(G), len(M)}) != 1:
        raise NumericalFailure("invalid_input", "Score array lengths differ")
    update_D, alpha = stable_projection(G, a)
    update_Y, beta = stable_projection(M, b)
    rD, rY = a - update_D, b - update_Y
    scores = _finite_vector(rD * rY - G * rD - M * rY, "corrected score")
    diagnostics = {"alpha": alpha["coefficient"], "beta": beta["coefficient"],
                   "G_direction_sum_squares": alpha["direction_sum_squares"],
                   "M_direction_sum_squares": beta["direction_sum_squares"],
                   "G_direction_norm": alpha["direction_norm"], "M_direction_norm": beta["direction_norm"],
                   "D_update_norm": alpha["update_norm"], "Y_update_norm": beta["update_norm"],
                   "zero_G": alpha["zero_direction"], "zero_M": beta["zero_direction"],
                   "mean_G_rD": float(np.mean(G*rD)), "mean_M_rY": float(np.mean(M*rY)),
                   "full_score_variance": float(np.var(scores, ddof=1)),
                   "product_only_variance": float(np.var(rD*rY, ddof=1))}
    return {"scores": scores, "rD": rD, "rY": rY, "diagnostics": diagnostics}


def dukes_from_initial(D, Y, theta, g_hat, m_hat, folds, kernel_config=None):
    D, Y, g_hat, m_hat = [_finite_vector(x, name) for x, name in
                         ((D,"D"),(Y,"Y"),(g_hat,"g_hat"),(m_hat,"m_hat"))]
    if len({len(D), len(Y), len(g_hat), len(m_hat)}) != 1 or not np.isfinite(theta):
        raise NumericalFailure("invalid_input", "Input dimensions/theta invalid")
    folds = validate_folds(folds, len(D))
    a, b = D-g_hat, Y-theta*D-m_hat
    arrays = {key: np.empty(len(D)) for key in ("G", "M", "rD", "rY")}
    scores, diagnostics = np.empty(len(D)), []
    for k, (_, test) in enumerate(folds):
        try:
            G, diag_G = select_nw(g_hat[test], b[test], kernel_config)
            M, diag_M = select_nw(m_hat[test], a[test], kernel_config)
            out = corrected_score(a[test], b[test], G, M)
            scores[test] = out["scores"]
            for key, value in (("G",G),("M",M),("rD",out["rD"]),("rY",out["rY"])):
                arrays[key][test] = value
            diag = {"fold": k, "n_fold": len(test), **out["diagnostics"]}
            for suffix, part in (("G", diag_G), ("M", diag_M)):
                diag.update({f"{name}_{suffix}": value for name, value in part.items()})
            diagnostics.append(diag)
        except NumericalFailure as exc:
            exc.fold = k
            raise
    return {"scores": scores, "standard_scores": a*b,
            "fold_diagnostics": diagnostics, "arrays": {**arrays, "a": a, "b": b}}


## 11. M3：一步修正与配套 Wald 适配

令 theta_a 为 M1 初值，g 为共享的处理 OOF 预测，只在 theta_a 处学习 m_a(X)=E[Y−theta_a D|X]，每折一个模型。
在各评价折用原 NW 规则计算 G、M，校准 alpha、beta。记 w=D−g，a=w−alpha G，b_a=Y−theta_a D−m_a−beta M。
冻结 G、M、alpha，并以 m(theta)=m_a−(theta−theta_a)g 线性化 nuisance；beta 随 theta 的投影变化由解析式处理。
令 v=w−Proj_M(w)，则 b(theta)=b_a−(theta−theta_a)v，完整 score 为 psi(theta)=a b(theta)−G a−M b(theta)。

J=mean((a−M)v)，theta_hat=theta_a+mean(psi(theta_a))/J。
SE=sqrt(mean((psi(theta_hat)−mean(psi(theta_hat)))^2)/(n J^2))，区间 theta_hat ± z SE。
这是对已声明冻结路径的解析根和工作 sandwich；不是原文在每个 theta 重训后的 score 根，不声称完整计入初值、辅助模型和带宽估计的一阶影响。
无迭代、无端点搜索、无模型重训来计算区间。J 退化或条件数过大则明确失败，不回退 M1。


In [ ]:
def m3_one_step(observed, folds, config, dgp, error, rep, mean_predictions):
    """User-authorized frozen-path adaptation, not the original Dukes inversion."""
    X, D, Y = observed
    g, ell = mean_predictions
    w = D-g
    anchor = ols_point(w, Y-ell)
    m_anchor = np.empty(len(D))
    for fold_id, (train, test) in enumerate(folds):
        model = learner(config, seed(config, dgp, error, rep, 6, fold_id, 1))
        m_anchor[test] = model.fit(X[train], (Y-anchor*D)[train]).predict(X[test])
    if not np.isfinite(m_anchor).all():
        raise ArithmeticError('M3_nonfinite_anchor_nuisance')
    original = dukes_from_initial(D,Y,anchor,g,m_anchor,folds,config['spec']['kernel'])
    G, M = original['arrays']['G'], original['arrays']['M']
    a, b_anchor = original['arrays']['rD'], original['arrays']['rY']
    v = np.empty(len(D))
    for fold_id, (_, test) in enumerate(folds):
        update, _ = stable_projection(M[test], w[test])
        v[test] = w[test]-update
    derivative = (a-M)*v
    J = float(np.mean(derivative))
    condition = float(np.mean(abs(derivative))/max(abs(J), np.finfo(float).tiny))
    if not np.isfinite(J) or abs(J)<=SOLVER['min_denominator']:
        raise ArithmeticError(f'M3_degenerate_working_bread: J={J}')
    if not np.isfinite(condition) or condition>SOLVER['m3_condition_limit']:
        raise ArithmeticError(f'M3_ill_conditioned_working_bread: condition={condition}')
    initial_score = original['scores']
    delta = float(np.mean(initial_score)/J)
    theta = float(anchor+delta)
    b_final = b_anchor-delta*v
    score = a*b_final-G*a-M*b_final
    # Separate algebraic checks: full score identity, root residual and calibration.
    alternate = initial_score-delta*derivative
    if not np.allclose(score,alternate,rtol=1e-10,atol=1e-10):
        raise ArithmeticError('M3_linear_path_score_identity')
    residual = float(abs(np.mean(score)))
    if not np.isfinite(theta) or residual>SOLVER['score_tol']*max(1.,float(np.std(score))):
        raise ArithmeticError('M3_linear_path_root_residual')
    moments = []
    for _,test in folds:
        moments.append(dict(mean_G_a=float(np.mean(G[test]*a[test])),
                            mean_M_b_final=float(np.mean(M[test]*b_final[test]))))
    return dict(theta_hat=theta, se_type='M3_frozen_path_plugin_ddof0',
                ci_type='M3_one_step_plugin_Wald', _working_scores=score, _working_J=J,
                diagnostics=dict(anchor=anchor, correction=delta, J=J,
                    working_bread_condition=condition, root_score_residual=residual,
                    nuisance_candidate_count=1, transformed_outcome_fits=5,
                    auxiliary_kernel_regressions=10, interval_model_fits=0,
                    inference_scope='Working sandwich for the frozen linear path; unconditional DR validity not established.',
                    fold_moments=moments, fold_diagnostics=original['fold_diagnostics']))


## 12. 四方法统一 Wald 接口与分段计时

M1/M4 共享均值预测；M2 学习中位数；M3 复用处理预测并额外进行 5 次初值处的变换结果拟合。
每方法分别记录 point_elapsed_seconds、ci_elapsed_seconds；M3 的区间阶段只使用已保存的 score 与 J，不进行任何拟合。
M3 的方法标签和 se_type/ci_type 明确注明 one-step/frozen-path plug-in，不能与旧反演结果混用。


In [ ]:
def estimate_method(index, observed, folds, config, dgp, error, rep, cache):
    point_start = time.perf_counter()
    target = 'median' if index==1 else 'mean'
    if target not in cache:
        cache[target] = crossfit(observed,folds,config,dgp,error,rep,median=index==1)
    predictions = cache[target]
    w,u = observed[1]-predictions[0], observed[2]-predictions[1]
    if index==2:
        out=m3_one_step(observed,folds,config,dgp,error,rep,predictions)
    else:
        out=(m1 if index==0 else m2 if index==1 else m4)(w,u)
        out['diagnostics']['prediction_hash']=sha(predictions.tobytes())
    out['point_elapsed_seconds']=time.perf_counter()-point_start
    ci_start=time.perf_counter()
    if index==2:
        score=out.pop('_working_scores'); J=out.pop('_working_J')
        variance=float(np.mean((score-score.mean())**2))
        out['se']=float(np.sqrt(variance/(len(score)*J*J))) if np.isfinite(variance) and variance>0 else None
        out['diagnostics']['working_score_variance_ddof0']=variance
    se=out.get('se')
    valid=se is not None and math.isfinite(se) and se>0
    z=1.96 if config['alpha']==.05 else float(norm.ppf(1-config['alpha']/2))
    t=out['theta_hat']
    out.update(ci_components=[[t-z*se,t+z*se]] if valid else [],
               ci_status='ok' if valid else 'ci_failed')
    out['ci_elapsed_seconds']=time.perf_counter()-ci_start
    return out


## 13. 逐方法 checkpoint 与主循环

每个方法结束即保存。新版签名与旧版不同，不读取旧反演记录。M3 没有候选网格或端点循环。
full 入口要求同版本 smoke 报告通过且依赖匹配，实际 n/R/K/网格严格固定；不会自动启动 full。
RESUME 跳过同签名已落盘终态，保留失败；纯调度开关不改变统计签名。N_JOBS=1。


In [ ]:
def run(config, resume=True, retry_failed=False, stop_after_reps=None, n_jobs=1):
    if n_jobs != 1:
        raise ValueError('v1 only supports N_JOBS=1; parallel execution has not been validated')
    if retry_failed:
        raise ValueError('v1 preserves all terminal failures; revise/version code before retrying')
    folder = Path(config['project_root'])/'work'/config['stage_id']/config['mode']/config['run_id']
    folder.mkdir(parents=True, exist_ok=True)
    if config['mode'] == 'full':
        assert config['n']==1000 and config['R']==1000 and config['K']==5 and config['p']==5
        assert config['errors']==list(range(8))
        smoke_root=Path(config['project_root'])/'work'/config['stage_id']/'smoke'
        passed=False
        for report_path in smoke_root.glob('*/validation_report.json'):
            report=json.loads(report_path.read_text('utf-8'))
            if (report.get('status')=='SMOKE_PASSED'
                and report.get('method_signature')==config['method_signature']
                and report.get('dependencies')==config['dependencies']):
                passed=True
        if not passed:
            raise RuntimeError('本版本尚无通过的 smoke：先运行 smoke 并完成验证，再由用户切换 full。')
    saved = folder/'config.json'
    if saved.exists() and json.loads(saved.read_text('utf-8')) != config:
        raise RuntimeError('Checkpoint configuration mismatch; choose a new version/run')
    atomic_json(saved, config)
    atomic_json(folder/'manifest.json', dict(config_hash=config['config_hash'], versions=versions(),
                method_signature=config['method_signature'], system=platform.platform(),
                status='user_selected_'+config['mode'], scheduler=dict(resume=resume,
                retry_failed=retry_failed, stop_after_reps=stop_after_reps, n_jobs=n_jobs)))
    attempted = 0
    lock = folder/'.running.lock'
    try:
        handle = lock.open('x', encoding='utf-8')
    except FileExistsError as exc:
        raise RuntimeError('A run lock exists; confirm the old kernel is stopped before removing it') from exc
    handle.write(str(os.getpid())); handle.close()
    try:
        with threadpool_limits(limits=1):
            for dgp in range(2):
                for error in config['errors']:
                    for rep in range(config['R']):
                        stem = f'd{dgp}_e{error}_r{rep:04d}'
                        paths = [folder/'records'/f'{stem}_{i}.json' for i in range(4)]
                        for method_index, candidate_path in enumerate(paths):
                            if candidate_path.exists():
                                saved_row = json.loads(candidate_path.read_text('utf-8'))
                                required = dict(config_hash=config['config_hash'],
                                                dgp=DGP_NAMES[dgp], error_id=error,
                                                replication_id=rep, method_id=METHODS[method_index])
                                if any(saved_row.get(key) != value for key, value in required.items()):
                                    raise RuntimeError(f'Checkpoint identity mismatch: {candidate_path}')
                        if all(p.exists() for p in paths):
                            if resume:
                                continue
                            raise RuntimeError('RESUME=False with existing records; do not overwrite old attempts')
                        if stop_after_reps is not None and attempted >= stop_after_reps:
                            return folder
                        observed, folds, ids, audit = generate_observed(config, dgp, error, rep)
                        base = dict(run_id=config['run_id'], stage_id=config['stage_id'], mode=config['mode'],
                            p=5, n=config['n'], K=5, dgp=DGP_NAMES[dgp], error_id=error,
                            error_family='gaussian' if error==0 else 'student_t',
                            error_parameter=None if error==0 else T_GRID[error-1], replication_id=rep,
                            estimator_signature=config['method_signature'], config_hash=config['config_hash'],
                            code_hash=config['code_hash'], data_seed=audit['seeds']['X'],
                            fold_seed=audit['seeds']['folds'], data_hash=audit['data_hash'],
                            fold_hash=audit['fold_hash'], seeds=audit['seeds'], attempt_id=1)
                        for saved_path in paths:
                            if saved_path.exists():
                                previous = json.loads(saved_path.read_text('utf-8'))
                                if previous['data_hash'] != audit['data_hash'] or previous['fold_hash'] != audit['fold_hash']:
                                    raise RuntimeError('Resumed data/fold fingerprints differ')
                        cache = {}
                        for i, record_path in enumerate(paths):
                            if record_path.exists() and resume:
                                continue
                            print(f'{stem} | starting {METHODS[i]}', flush=True)
                            start = time.perf_counter()
                            row = dict(base, method_id=METHODS[i], implementation_variant=VARIANTS[i],
                                       theta_hat=None, se=None, se_type=None, ci_type=None, ci_components=[],
                                       point_status='point_failed', ci_status='ci_failed',
                                       point_elapsed_seconds=None, ci_elapsed_seconds=None)
                            with warnings.catch_warnings(record=True) as caught:
                                warnings.simplefilter('always')
                                try:
                                    if i in (2,3) and 'mean' not in cache:
                                        cache['mean'] = crossfit(observed, folds, config, dgp, error, rep)
                                    out = estimate_method(i, observed, folds, config, dgp, error, rep, cache)
                                    row.update(out, point_status='ok')
                                except Exception as exc:
                                    row.update(exception_type=type(exc).__name__, error=str(exc),
                                               traceback=traceback.format_exc(), failure_location=METHODS[i])
                            row['warnings'] = [str(w.message) for w in caught]
                            row['elapsed_seconds'] = time.perf_counter()-start
                            row['status'] = 'point_failed' if row['point_status'] != 'ok' else row['ci_status']
                            valid_ci = row['ci_status'] == 'ok'
                            components = row['ci_components']
                            row.update(ci_lower=components[0][0] if valid_ci and components else None,
                                       ci_upper=components[-1][1] if valid_ci and components else None,
                                       ci_length=sum(b-a for a,b in components) if valid_ci else None)
                            diagnostic = row.pop('diagnostics', {})
                            diag_path = folder/'diagnostics'/f'{stem}_{i}.json'
                            atomic_json(diag_path, diagnostic)
                            row['diagnostics_ref'] = str(diag_path.relative_to(folder))
                            atomic_json(record_path, row)  # persist each method before the next slow method
                            if row['status'] != 'ok':
                                with (folder/'failures.jsonl').open('a', encoding='utf-8') as stream:
                                    stream.write(json.dumps(row, ensure_ascii=False)+'\n')
                        attempted += 1
                        records = load_records(folder)
                        ok = sum(r['status']=='ok' for r in records)
                        planned = 2*len(config['errors'])*config['R']*4
                        print(f'{stem}: planned={planned}, attempted={len(records)}, ok={ok}, '
                              f'failed/unresolved={len(records)-ok}, pending={planned-len(records)}', flush=True)
    finally:
        lock.unlink(missing_ok=True)
    return folder

def load_records(folder):
    return [json.loads(p.read_text('utf-8')) for p in sorted((Path(folder)/'records').glob('*.json'))]


## 14. 评价与内部图表

真值只在此评价步骤进入。点估计有效与 CI 有效分母分开；M3 未解出的集合不会算作未覆盖。
coverage deterioration 保留负值，rmse_ratio 的基准是同一 DGP/方法的 Gaussian。
配对 coverage 差按共同有效 replication 计算；R=3 的数值不用于论文研究结论。
开发图和 summary 只写入本次 smoke 的 work 目录。


In [ ]:
def summarize(folder, config):
    rows = load_records(folder)
    evaluated = []
    for r in rows:
        q = r.copy()
        q['theta_true'] = 1.
        q['estimation_error'] = r['theta_hat']-1 if r['point_status']=='ok' else None
        q['covered'] = any(a<=1<=b for a,b in r['ci_components']) if r['ci_status']=='ok' else None
        evaluated.append(q)
    frame = pd.DataFrame(evaluated)
    if frame.empty:
        return frame
    frame.to_csv(Path(folder)/'evaluated_records.csv', index=False, encoding='utf-8-sig')
    summaries = []
    for (dgp,e,method), group in frame.groupby(['dgp','error_id','method_id']):
        point = group[group.point_status=='ok']
        ci = group[group.ci_status=='ok']
        same = point[point.se.notna() & (point.se>0)]
        n, nc = len(point), len(ci)
        errors = point.estimation_error.astype(float).to_numpy()
        rmse = float(np.sqrt(np.mean(errors**2))) if n else None
        coverage = float(ci.covered.astype(float).mean()) if nc else None
        sd = float(point.theta_hat.std(ddof=1)) if n>1 else None
        common_sd = float(same.theta_hat.std(ddof=1)) if len(same)>1 else None
        meanse = float(same.se.mean()) if len(same) else None
        row = dict(mode=config['mode'], p=5, dgp=dgp, error_id=int(e), method_id=method,
            estimator_signature=config['method_signature'], R_planned=config['R'], R_attempted=len(group),
            N_point_valid=n, N_ci_valid=nc, N_se_valid=len(same),
            N_point_ci_common=int(((group.point_status=='ok')&(group.ci_status=='ok')).sum()),
            bias=float(errors.mean()) if n else None, rmse=rmse, empirical_sd=sd,
            mean_se=meanse, se_common_sd=common_sd,
            se_calibration=meanse/common_sd if common_sd and meanse else None,
            coverage=coverage, coverage_mcse=math.sqrt(coverage*(1-coverage)/nc) if nc else None,
            bias_mcse=sd/math.sqrt(n) if sd is not None else None,
            rmse_mcse=float(np.std(errors**2,ddof=1)/(2*rmse*math.sqrt(n))) if n>1 and rmse else None,
            mean_ci_length=float(ci.ci_length.mean()) if nc else None,
            median_ci_length=float(ci.ci_length.median()) if nc else None,
            failure_rate=float((group.status!='ok').mean()),
            coverage_missing_lower=float(ci.covered.astype(float).sum()/config['R']),
            coverage_missing_upper=float((ci.covered.astype(float).sum()+config['R']-nc)/config['R']),
            quality_status='INCOMPLETE_CANDIDATE')
        summaries.append(row)
    result = pd.DataFrame(summaries)
    # Entire groups may fail: convert missing metrics to numeric NaN before arithmetic.
    for column in ('coverage', 'rmse', 'bias', 'empirical_sd', 'mean_se',
                   'se_common_sd', 'se_calibration', 'coverage_mcse', 'bias_mcse',
                   'rmse_mcse', 'mean_ci_length', 'median_ci_length'):
        result[column] = pd.to_numeric(result[column], errors='coerce')
    for i,r in result.iterrows():
        base = result[(result.dgp==r.dgp)&(result.method_id==r.method_id)&(result.error_id==0)]
        if len(base):
            b = base.iloc[0]
            result.loc[i,'coverage_deterioration_pp'] = 100*(b.coverage-r.coverage)
            result.loc[i,'rmse_ratio'] = r.rmse/b.rmse if pd.notna(b.rmse) and b.rmse>0 else np.nan
    result.to_csv(Path(folder)/'summary.csv',index=False,encoding='utf-8-sig')
    return result


def plot_smoke(folder, summary):
    """Exploratory figures only. An unresolved CI has no reported coverage."""
    import matplotlib.pyplot as plt
    metrics = [('coverage','Coverage'), ('coverage_deterioration_pp','Coverage deterioration (pp)'),
               ('bias','Bias'), ('rmse','RMSE'), ('rmse_ratio','RMSE / Gaussian RMSE'),
               ('mean_se','Mean SE'), ('empirical_sd','Empirical SD'),
               ('se_calibration','SE calibration'), ('mean_ci_length','CI length'),
               ('failure_rate','Failure/unresolved rate')]
    output = Path(folder)/'figures'
    output.mkdir(exist_ok=True)
    for metric, label in metrics:
        fig, axes = plt.subplots(1,2,figsize=(11,4), squeeze=False)
        for dgp, ax in zip(DGP_NAMES, axes[0]):
            for method in METHODS:
                part = summary[(summary.dgp==dgp)&(summary.method_id==method)].sort_values('error_id')
                if metric not in part:
                    continue
                ax.plot(range(len(part)), part[metric], marker='o',label=method)
                ax.set_xticks(range(len(part)), ['Gaussian' if e==0 else f't({T_GRID[int(e)-1]:g})' for e in part.error_id])
            if metric == 'coverage': ax.axhline(.95,color='black',ls='--')
            ax.set_title(dgp+' | '+str(summary['mode'].iloc[0])); ax.set_ylabel(label)
            ax.legend(fontsize=7); ax.grid(alpha=.2)
        fig.tight_layout(); fig.savefig(output/(metric+'.png'),dpi=160); plt.close(fig)


def paired_diagnostics(folder):
    frame = pd.read_csv(Path(folder)/'evaluated_records.csv')
    rows = []
    for (dgp,e), group in frame.groupby(['dgp','error_id']):
        for other in METHODS[:3]:
            a = group[(group.method_id==METHODS[3])&(group.ci_status=='ok')].set_index('replication_id')
            b = group[(group.method_id==other)&(group.ci_status=='ok')].set_index('replication_id')
            shared = a.index.intersection(b.index)
            delta = a.loc[shared,'covered'].astype(float)-b.loc[shared,'covered'].astype(float)
            rows.append(dict(dgp=dgp,error_id=int(e),reference=other,N_pair=len(shared),
                             coverage_M4_minus_reference=float(delta.mean()) if len(shared) else None,
                             coverage_difference_mcse=float(delta.std(ddof=1)/np.sqrt(len(shared))) if len(shared)>1 else None))
    pd.DataFrame(rows).to_csv(Path(folder)/'paired_diagnostics.csv',index=False,encoding='utf-8-sig')


## 15. 用户执行的验证与复现

检查 72 个 smoke 方法键、状态、数据/fold 一致性、M1/M4 预测一致、Wald 端点与 SE、M3 解析方程残差。
用户执行的 replay 会独立重建一份相同 seed 的数据与四方法，比较点估计/SE/区间，并核对恢复不会覆盖完整 checkpoint。
这些检查验证实现和可复现性，不证明 Wald 在各 DGP 下的渐近覆盖率。不会以 coverage 接近 95% 或某方法胜出为通过条件。


In [ ]:
def verify_replay(folder,config):
    """User executes one repeated dataset plus a completed-run resume no-op."""
    rows=load_records(folder)
    planned=2*len(config['errors'])*config['R']*4
    if len(rows)!=planned or any(r['status']!='ok' for r in rows):
        report=dict(passed=False,reason='Requires a complete successful run before reproducibility verification')
        atomic_json(Path(folder)/'replay_report.json',report)
        return report
    observed,folds,ids,audit=generate_observed(config,0,0,0)
    cache={}; comparisons=[]
    with threadpool_limits(limits=1):
        for i,method in enumerate(METHODS):
            out=estimate_method(i,observed,folds,config,0,0,0,cache)
            prior=next(r for r in rows if r['dgp']==DGP_NAMES[0] and r['error_id']==0
                       and r['replication_id']==0 and r['method_id']==method)
            comparisons.append(dict(method=method,
                matched=bool(prior['data_hash']==audit['data_hash'] and prior['fold_hash']==audit['fold_hash']
                  and np.allclose([out['theta_hat'],out['se']],[prior['theta_hat'],prior['se']],rtol=1e-10,atol=1e-10)
                  and np.allclose(out['ci_components'],prior['ci_components'],rtol=1e-10,atol=1e-10))))
    before={p.name:sha(p.read_bytes()) for p in (Path(folder)/'records').glob('*.json')}
    run(config,resume=True)
    after={p.name:sha(p.read_bytes()) for p in (Path(folder)/'records').glob('*.json')}
    report=dict(passed=all(c['matched'] for c in comparisons) and before==after,
                comparisons=comparisons,complete_checkpoint_replay_unchanged=before==after,
                scope='Same-seed independent regeneration and completed-checkpoint replay. For interrupted-run check use STOP_AFTER_REPS=2 then resume.')
    atomic_json(Path(folder)/'replay_report.json',report)
    return report

def validate(folder,config):
    rows=load_records(folder)
    keys=[(r['dgp'],r['error_id'],r['replication_id'],r['method_id']) for r in rows]
    expected={(d,e,r,m) for d in DGP_NAMES for e in config['errors'] for r in range(config['R']) for m in METHODS}
    checks=dict(exact_task_keys=set(keys)==expected,unique=len(keys)==len(set(keys)),
        all_methods_success=len(rows)>0 and all(r['status']=='ok' for r in rows),
        matching_config=all(r['config_hash']==config['config_hash'] for r in rows))
    z=1.96 if config['alpha']==.05 else float(norm.ppf(1-config['alpha']/2))
    for row in rows:
        key=f"{row['dgp']}_{row['error_id']}_{row['replication_id']}_{row['method_id']}"
        if row['status']=='ok':
            t,se=row['theta_hat'],row['se']
            checks['Wald_'+key]=bool(se is not None and se>0 and math.isfinite(se)
                and np.allclose(row['ci_components'],[[t-z*se,t+z*se]],rtol=1e-10,atol=1e-10))
        else: checks['Wald_'+key]=False
        if row['method_id']==METHODS[2] and row['point_status']=='ok':
            diag=json.loads((Path(folder)/row['diagnostics_ref']).read_text('utf-8'))
            checks['M3_algebra_'+key]=bool(diag['root_score_residual']<=SOLVER['score_tol']
                and diag['nuisance_candidate_count']==1 and diag['interval_model_fits']==0
                and row['ci_type']=='M3_one_step_plugin_Wald')
    for dgp in DGP_NAMES:
        for e in config['errors']:
            for rep in range(config['R']):
                group=[r for r in rows if (r['dgp'],r['error_id'],r['replication_id'])==(dgp,e,rep)]
                if len(group)==4:
                    label=f'{dgp}_{e}_{rep}'
                    checks['shared_data_'+label]=len({r['data_hash'] for r in group})==1
                    checks['shared_folds_'+label]=len({r['fold_hash'] for r in group})==1
                    hashes=[]
                    for r in group:
                        if r['method_id'] in (METHODS[0],METHODS[3]) and r['point_status']=='ok':
                            diag=json.loads((Path(folder)/r['diagnostics_ref']).read_text('utf-8'))
                            hashes.append(diag.get('prediction_hash'))
                    checks['shared_M1_M4_predictions_'+label]=len(hashes)==2 and hashes[0]==hashes[1]
    replay_path=Path(folder)/'replay_report.json'
    checks['reproducible_checkpoint_replay']=replay_path.exists() and json.loads(replay_path.read_text('utf-8')).get('passed',False)
    complete=all(checks.values())
    report=dict(status=('SMOKE_PASSED' if config['mode']=='smoke' else 'FULL_READY_FOR_REVIEW') if complete else 'NOT_PASSED',
        mode=config['mode'],records=len(rows),expected_records=len(expected),checks=checks,
        method_signature=config['method_signature'],config_hash=config['config_hash'],dependencies=config['dependencies'],
        note='M3 is a user-authorized one-step/frozen-path plug-in Wald adaptation; computational checks do not establish its inferential validity.')
    atomic_json(Path(folder)/'validation_report.json',report)
    report_text='# Stage 1 validation: '+report['status']+'\n\n'+report['note']+'\n\n'
    report_text+='\n'.join(f'- {name}: {value}' for name,value in checks.items())
    (Path(folder)/'validation_report.md').write_text(report_text,encoding='utf-8')
    return report

def static_source_check(config):
    checks={}
    for item in SOURCE_INVENTORY:
        original=Path(item['path'])
        if original.exists():
            checks[item['id']]='original_hash_verified' if sha(original.read_bytes())==item['sha256'] else 'original_changed_since_inventory'
        else: checks[item['id']]='original_unavailable_embedded_provenance_retained'
    for median in (False,True): learner(config,0,median=median)._validate_params()
    checks['learner_parameters']='mean_and_median_validated_without_fit'
    checks['M3']='one_anchor_fit_set; analytic_one_step; Wald_only'
    return checks


In [ ]:
IMPLEMENTATION_NAMES = ('locate_project_root',
 'sha',
 'digest',
 'atomic_json',
 'versions',
 'implementation_fingerprint',
 'configuration',
 'ar1_covariance',
 'nuisance_functions',
 'standardized_student_t',
 'seed',
 'generate_observed',
 'learner',
 'crossfit',
 'ols_point',
 'm1',
 'm2',
 'm4',
 'NumericalFailure',
 '_finite_vector',
 'validate_folds',
 '_nw_from_distances',
 'nw_predict',
 'select_nw',
 'stable_projection',
 'corrected_score',
 'dukes_from_initial',
 'm3_one_step',
 'estimate_method',
 'run',
 'load_records',
 'summarize',
 'plot_smoke',
 'paired_diagnostics',
 'verify_replay',
 'validate',
 'static_source_check')


CONFIG=configuration(PROJECT_ROOT,MODE,BASE_SEED,ALPHA)
print('模式与规模：',CONFIG['mode'],CONFIG['n'],CONFIG['R'],CONFIG['K'])
print('M3 variant:',VARIANTS[2])
print('方法签名：',CONFIG['method_signature'])
print('运行签名：',CONFIG['config_hash'])
print('依赖：',CONFIG['dependencies'])
display(static_source_check(CONFIG))


## 加速补丁：并行调度与增量进度（2026-11-04）

统计函数、学习器、seed 和已接受配置保持原样。为兼容当前已完成结果，原 v4 定义保留为签名基线；下方调度补丁在 CONFIG 构造后接管 run/validate。
不再每完成一份数据重读全部结果；只在启动时扫描一次，之后增量计数。验证分组也改为一次建立索引。
默认 N_JOBS=4，独立数据重复并行，每进程1线程。每个任务独占记录文件；失败日志由主进程写入。
每10份数据输出一次进度和本次ETA，减少界面输出开销。

**磁盘修改不会改变正在运行的旧循环。请用户点击“中断”，等当前运行停止，再重启内核，重新打开保存后的 Notebook，从头 Run All。**
默认 MODE=full、RESUME=True，将继续现有 full_a063bc681962059a。不要运行旧的未刷新标签页，不要重跑已完成数据。
如果强制重启后残留 .running.lock，必须先确认旧进程停止再处理；代码不会绕过活跃锁。
所有模型、并行一致性检查、smoke/full 仍由用户执行，Codex没有代运行。


In [ ]:
# This cell runs AFTER CONFIG was built from the accepted v4 baseline definitions.
# It changes scheduling/I/O only. Statistical functions, config, seeds and run_id stay identical.
from joblib import Parallel, delayed, parallel_config
SCHEDULER_REVISION='v4_scheduler_20261104_parallel_incremental'
PROGRESS_EVERY_REPS=10
_ACCEPTED_SERIAL_RUN=run
_ACCEPTED_VALIDATE=validate

def _persist_replication(config,task,folder_text,persist=True):
    dgp,error,rep=task
    folder=Path(folder_text)
    stem=f'd{dgp}_e{error}_r{rep:04d}'
    started=time.perf_counter()
    paths=[folder/'records'/f'{stem}_{i}.json' for i in range(4)]
    with threadpool_limits(limits=1):
        observed,folds,ids,audit=generate_observed(config,dgp,error,rep)
    base=dict(run_id=config['run_id'],stage_id=config['stage_id'],mode=config['mode'],
        p=5,n=config['n'],K=5,dgp=DGP_NAMES[dgp],error_id=error,
        error_family='gaussian' if error==0 else 'student_t',
        error_parameter=None if error==0 else T_GRID[error-1],replication_id=rep,
        estimator_signature=config['method_signature'],config_hash=config['config_hash'],
        code_hash=config['code_hash'],data_seed=audit['seeds']['X'],fold_seed=audit['seeds']['folds'],
        data_hash=audit['data_hash'],fold_hash=audit['fold_hash'],seeds=audit['seeds'],attempt_id=1,
        scheduler_revision=SCHEDULER_REVISION)
    cache={}; written=[]
    with threadpool_limits(limits=1):
        for i,record_path in enumerate(paths):
            if persist and record_path.exists():
                prior=json.loads(record_path.read_text('utf-8'))
                if (prior['data_hash']!=audit['data_hash'] or prior['fold_hash']!=audit['fold_hash']
                    or prior['config_hash']!=config['config_hash'] or prior['method_id']!=METHODS[i]):
                    raise RuntimeError('Resumed record identity differs: '+str(record_path))
                continue
            start=time.perf_counter()
            row=dict(base,method_id=METHODS[i],implementation_variant=VARIANTS[i],
                theta_hat=None,se=None,se_type=None,ci_type=None,ci_components=[],
                point_status='point_failed',ci_status='ci_failed',
                point_elapsed_seconds=None,ci_elapsed_seconds=None)
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter('always')
                try:
                    # The exact accepted estimator dispatch is called unchanged.
                    out=estimate_method(i,observed,folds,config,dgp,error,rep,cache)
                    row.update(out,point_status='ok')
                except Exception as exc:
                    row.update(exception_type=type(exc).__name__,error=str(exc),
                               traceback=traceback.format_exc(),failure_location=METHODS[i])
            row['warnings']=[str(w.message) for w in caught]
            row['elapsed_seconds']=time.perf_counter()-start
            row['status']='point_failed' if row['point_status']!='ok' else row['ci_status']
            valid=row['ci_status']=='ok'; components=row['ci_components']
            row.update(ci_lower=components[0][0] if valid and components else None,
                       ci_upper=components[-1][1] if valid and components else None,
                       ci_length=sum(b-a for a,b in components) if valid else None)
            diagnostic=row.pop('diagnostics',{})
            diag_path=folder/'diagnostics'/f'{stem}_{i}.json'
            row['diagnostics_ref']=str(diag_path.relative_to(folder))
            if persist:
                # Each task owns disjoint filenames; each method is saved immediately.
                atomic_json(diag_path,diagnostic)
                atomic_json(record_path,row)
            written.append(row)
    return dict(stem=stem,seconds=time.perf_counter()-started,
                count=len(written),ok=sum(r['status']=='ok' for r in written),
                failures=[r for r in written if r['status']!='ok'],
                replay_rows=written if not persist else [])

def _scheduler_audit():
    names=('_persist_replication','_run_incremental','_validate_linear','verify_parallel_replay')
    source={name:ast.dump(ast.parse(textwrap.dedent(inspect.getsource(globals()[name]))),
                          include_attributes=False) for name in names}
    return dict(revision=SCHEDULER_REVISION,execution_source_hash=digest(source),
        compatibility='Legacy code_hash identifies accepted v4 baseline; execution_source_hash records this scheduler patch. Estimator definitions and run config are unchanged.')

def _run_incremental(config,resume=True,retry_failed=False,stop_after_reps=None,n_jobs=1):
    if not isinstance(n_jobs,int) or not 1<=n_jobs<=max(1,os.cpu_count() or 1):
        raise ValueError('N_JOBS must be between 1 and the available logical CPU count')
    if retry_failed:
        raise ValueError('Terminal failures are preserved. Do not retry with altered statistical settings.')
    if stop_after_reps is not None and (not isinstance(stop_after_reps,int) or stop_after_reps<0):
        raise ValueError('STOP_AFTER_REPS must be None or a nonnegative integer')
    folder=Path(config['project_root'])/'work'/config['stage_id']/config['mode']/config['run_id']
    folder.mkdir(parents=True,exist_ok=True)
    if config['mode']=='full':
        assert config['n']==1000 and config['R']==1000 and config['K']==5 and config['p']==5
        assert config['errors']==list(range(8))
        passed=False
        for report_path in (Path(config['project_root'])/'work'/config['stage_id']/'smoke').glob('*/validation_report.json'):
            report=json.loads(report_path.read_text('utf-8'))
            passed|=(report.get('status')=='SMOKE_PASSED'
                     and report.get('method_signature')==config['method_signature']
                     and report.get('dependencies')==config['dependencies'])
        if not passed: raise RuntimeError('Run and validate the same-signature smoke before full.')
    cfg_path=folder/'config.json'
    if cfg_path.exists() and json.loads(cfg_path.read_text('utf-8'))!=config:
        raise RuntimeError('Checkpoint config mismatch: existing results were not changed.')
    lock=folder/'.running.lock'
    try: handle=lock.open('x',encoding='utf-8')
    except FileExistsError as exc:
        raise RuntimeError('A run lock exists. Interrupt the previous run before restarting; do not run two writers.') from exc
    handle.write(str(os.getpid())); handle.close()
    started=time.perf_counter(); completed=0; initial_count=0; pending=[]
    try:
        atomic_json(cfg_path,config)
        # Exactly one scan at startup. No all-record scan occurs in the completion loop.
        index={}; ok=0; failed=0
        for path in (folder/'records').glob('*.json'):
            row=json.loads(path.read_text('utf-8'))
            key=(row['dgp'],row['error_id'],row['replication_id'],row['method_id'])
            if key in index: raise RuntimeError('Duplicate checkpoint key: '+str(key))
            if (row['config_hash']!=config['config_hash'] or row['estimator_signature']!=config['method_signature']
                or row['mode']!=config['mode'] or row['n']!=config['n'] or row['p']!=5 or row['K']!=5
                or row['dgp'] not in DGP_NAMES or row['error_id'] not in config['errors']
                or not 0<=row['replication_id']<config['R'] or row['method_id'] not in METHODS):
                raise RuntimeError('Checkpoint identity mismatch: '+str(path))
            index[key]=row
            if row['status']=='ok':ok+=1
            else:failed+=1
        initial_count=len(index); count=initial_count
        for dgp in range(2):
            for error in config['errors']:
                for rep in range(config['R']):
                    keys=[(DGP_NAMES[dgp],error,rep,m) for m in METHODS]
                    group=[index[k] for k in keys if k in index]
                    if len({r['data_hash'] for r in group})>1 or len({r['fold_hash'] for r in group})>1:
                        raise RuntimeError('Previously saved task data/folds are inconsistent')
                    if group and not resume:
                        raise RuntimeError('RESUME=False with existing results; nothing was overwritten')
                    if len(group)<4:pending.append((dgp,error,rep))
        if stop_after_reps is not None:pending=pending[:stop_after_reps]
        planned=2*len(config['errors'])*config['R']*4
        audit=_scheduler_audit()
        atomic_json(folder/'manifest.json',dict(config_hash=config['config_hash'],versions=versions(),
            method_signature=config['method_signature'],system=platform.platform(),
            status='user_selected_'+config['mode'],scheduler=dict(n_jobs=n_jobs,inner_threads=1,
                resume=resume,retry_failed=retry_failed,stop_after_reps=stop_after_reps),**audit))
        print(f'Resume: {count}/{planned} records; {len(pending)} pending datasets; workers={n_jobs}, inner_threads=1',flush=True)
        failures_path=folder/'failures.jsonl'
        # One parent writer; restore a complete failure log from the startup index once.
        initial_failures=[row for row in index.values() if row['status']!='ok']
        if initial_failures:
            temporary=failures_path.with_suffix('.jsonl.tmp')
            temporary.write_text(''.join(json.dumps(row,ensure_ascii=False)+'\n' for row in initial_failures),encoding='utf-8')
            os.replace(temporary,failures_path)
        del index
        if n_jobs>1 and pending:
            replay_path=folder/'parallel_replay_report.json'
            prior_replay=json.loads(replay_path.read_text('utf-8')) if replay_path.exists() else {}
            if not (prior_replay.get('passed') and prior_replay.get('config_hash')==config['config_hash']
                    and prior_replay.get('execution_source_hash')==audit['execution_source_hash']):
                print('Checking two fixed datasets: serial versus two processes, without overwriting checkpoints...',flush=True)
                parallel_check=verify_parallel_replay(folder,config,n_jobs=2)
                if not parallel_check['passed']:
                    raise RuntimeError('Parallel reproducibility check failed. Existing checkpoints are preserved; use N_JOBS=1 and inspect parallel_replay_report.json.')
        work_started=time.perf_counter()
        def receive(result):
            nonlocal count,ok,failed,completed
            count+=result['count'];ok+=result['ok'];failed+=len(result['failures']);completed+=1
            if result['failures']:
                with failures_path.open('a',encoding='utf-8') as stream:
                    for row in result['failures']:stream.write(json.dumps(row,ensure_ascii=False)+'\n')
            if count>planned:raise RuntimeError('More records than planned')
            if completed<=3 or completed%PROGRESS_EVERY_REPS==0 or completed==len(pending):
                elapsed=time.perf_counter()-work_started
                eta=(len(pending)-completed)*elapsed/completed if completed else 0
                print(f"{result['stem']}: planned={planned}, attempted={count}, ok={ok}, failed={failed}, "
                      f"pending={planned-count}; session_datasets={completed}/{len(pending)}, elapsed={elapsed/60:.1f}m, ETA={eta/60:.1f}m",flush=True)
        if n_jobs==1:
            for task in pending:receive(_persist_replication(config,task,str(folder)))
        elif pending:
            # loky processes plus explicit thread limits avoid nested oversubscription.
            with parallel_config(backend='loky',inner_max_num_threads=1):
                with Parallel(n_jobs=n_jobs,return_as='generator_unordered',pre_dispatch=n_jobs,batch_size=1) as pool:
                    for result in pool(delayed(_persist_replication)(config,task,str(folder)) for task in pending):
                        receive(result)
        with (folder/'execution_log.jsonl').open('a',encoding='utf-8') as stream:
            stream.write(json.dumps(dict(scheduler_revision=SCHEDULER_REVISION,n_jobs=n_jobs,
                inner_threads=1,initial_records=initial_count,final_records=count,
                datasets_computed=completed,wall_seconds=time.perf_counter()-started,
                execution_source_hash=audit['execution_source_hash']))+'\n')
        return folder
    finally:
        # Joblib's context aborts/joins in-flight workers before this lock is released.
        lock.unlink(missing_ok=True)

def verify_parallel_replay(folder,config,n_jobs=2):
    """User-run check only: compare one dataset serially and in two worker processes."""
    tasks=[(0,0,0),(0,0,1)]
    serial=[_persist_replication(config,t,str(folder),persist=False) for t in tasks]
    with parallel_config(backend='loky',inner_max_num_threads=1):
        parallel=Parallel(n_jobs=min(2,n_jobs))(delayed(_persist_replication)(config,t,str(folder),persist=False) for t in tasks)
    checks=[]
    for left,right in zip(serial,parallel):
        for a,b in zip(left['replay_rows'],right['replay_rows']):
            good=(a['status']==b['status']=='ok' and a['data_hash']==b['data_hash'] and a['fold_hash']==b['fold_hash']
                and np.allclose([a['theta_hat'],a['se']],[b['theta_hat'],b['se']],rtol=1e-10,atol=1e-10)
                and np.allclose(a['ci_components'],b['ci_components'],rtol=1e-10,atol=1e-10))
            checks.append(dict(stem=left['stem'],method=a['method_id'],matched=bool(good)))
    report=dict(passed=all(c['matched'] for c in checks),checks=checks,
                config_hash=config['config_hash'],execution_source_hash=_scheduler_audit()['execution_source_hash'],
                scope='Two user-run observed datasets, serial versus two processes, no checkpoint records overwritten')
    atomic_json(Path(folder)/'parallel_replay_report.json',report)
    return report


def _validate_linear(folder,config):
    rows=load_records(folder)
    keys=[(r['dgp'],r['error_id'],r['replication_id'],r['method_id']) for r in rows]
    expected={(d,e,r,m) for d in DGP_NAMES for e in config['errors'] for r in range(config['R']) for m in METHODS}
    checks=dict(exact_task_keys=set(keys)==expected,unique=len(keys)==len(set(keys)),
        all_methods_success=len(rows)>0 and all(r['status']=='ok' for r in rows),
        matching_config=all(r['config_hash']==config['config_hash'] for r in rows))
    z=1.96 if config['alpha']==.05 else float(norm.ppf(1-config['alpha']/2))
    for row in rows:
        key=f"{row['dgp']}_{row['error_id']}_{row['replication_id']}_{row['method_id']}"
        if row['status']=='ok':
            t,se=row['theta_hat'],row['se']
            checks['Wald_'+key]=bool(se is not None and se>0 and math.isfinite(se)
                and np.allclose(row['ci_components'],[[t-z*se,t+z*se]],rtol=1e-10,atol=1e-10))
        else: checks['Wald_'+key]=False
        if row['method_id']==METHODS[2] and row['point_status']=='ok':
            diag=json.loads((Path(folder)/row['diagnostics_ref']).read_text('utf-8'))
            checks['M3_algebra_'+key]=bool(diag['root_score_residual']<=SOLVER['score_tol']
                and diag['nuisance_candidate_count']==1 and diag['interval_model_fits']==0
                and row['ci_type']=='M3_one_step_plugin_Wald')
    grouped={}
    for row in rows:
        grouped.setdefault((row['dgp'],row['error_id'],row['replication_id']),[]).append(row)
    for dgp in DGP_NAMES:
        for e in config['errors']:
            for rep in range(config['R']):
                group=grouped.get((dgp,e,rep),[])
                if len(group)==4:
                    label=f'{dgp}_{e}_{rep}'
                    checks['shared_data_'+label]=len({r['data_hash'] for r in group})==1
                    checks['shared_folds_'+label]=len({r['fold_hash'] for r in group})==1
                    hashes=[]
                    for r in group:
                        if r['method_id'] in (METHODS[0],METHODS[3]) and r['point_status']=='ok':
                            diag=json.loads((Path(folder)/r['diagnostics_ref']).read_text('utf-8'))
                            hashes.append(diag.get('prediction_hash'))
                    checks['shared_M1_M4_predictions_'+label]=len(hashes)==2 and hashes[0]==hashes[1]
    replay_path=Path(folder)/'replay_report.json'
    checks['reproducible_checkpoint_replay']=replay_path.exists() and json.loads(replay_path.read_text('utf-8')).get('passed',False)
    complete=all(checks.values())
    report=dict(status=('SMOKE_PASSED' if config['mode']=='smoke' else 'FULL_READY_FOR_REVIEW') if complete else 'NOT_PASSED',
        mode=config['mode'],records=len(rows),expected_records=len(expected),checks=checks,
        method_signature=config['method_signature'],config_hash=config['config_hash'],dependencies=config['dependencies'],
        note='M3 is a user-authorized one-step/frozen-path plug-in Wald adaptation; computational checks do not establish its inferential validity.')
    atomic_json(Path(folder)/'validation_report.json',report)
    report_text='# Stage 1 validation: '+report['status']+'\n\n'+report['note']+'\n\n'
    report_text+='\n'.join(f'- {name}: {value}' for name,value in checks.items())
    (Path(folder)/'validation_report.md').write_text(report_text,encoding='utf-8')
    return report


# CONFIG was intentionally computed before replacing these two orchestration functions.
# A fresh kernel reproduces the exact accepted method_signature and run_id.
run=_run_incremental
validate=_validate_linear
if CONFIG['mode']=='full':
    expected=Path(CONFIG['project_root'])/'work/stage_01/full/full_a063bc681962059a/config.json'
    if expected.exists() and CONFIG['run_id']!='full_a063bc681962059a':
        raise RuntimeError('Checkpoint signature differs from the existing full run. Stop here; do not recompute or delete old results. Restart the kernel and run the saved notebook from the beginning.')
print('Incremental progress + multiprocess scheduler enabled; estimator signatures unchanged. N_JOBS=',N_JOBS)


## 16. 用户运行所选模式

从新内核 Run All，默认只跑 smoke。M3 不再反演，也不迭代求根。Codex 没有代运行。
若需要检查恢复，先把 STOP_AFTER_REPS 设为 2 运行，再改回 None 重启内核运行，确认最终恰好 72 条记录。


In [ ]:
RUN_DIR = run(CONFIG, resume=RESUME, retry_failed=RETRY_FAILED,
              stop_after_reps=STOP_AFTER_REPS, n_jobs=N_JOBS)
print("本次输出：", RUN_DIR)


In [ ]:
SUMMARY=summarize(RUN_DIR,CONFIG)
if not SUMMARY.empty:
    display(SUMMARY)
    paired_diagnostics(RUN_DIR)
    plot_smoke(RUN_DIR,SUMMARY)
# User execution only: one repeat dataset and a checkpoint no-op, no extra Monte Carlo grid.
REPLAY=verify_replay(RUN_DIR,CONFIG)
REPORT=validate(RUN_DIR,CONFIG)
print('验证状态：',REPORT['status'])
print('报告：',RUN_DIR/'validation_report.md')
print('失败日志：',RUN_DIR/'failures.jsonl')


## 用户下一步

默认 MODE=smoke，重启内核后 Run All。核验 validation_report.md/json、replay_report.json 和失败日志。
新 M3 的点估计与区间耗时会分别记录；只有拿到新 smoke 才能估算正式 full 时间。
本次统计方法改变已在 variant、CI/SE 标签和签名中记录，旧版结果不能混入。
当前待用户 smoke；未执行新实验，未预填结果；不会自动运行 full，也没有创建后三阶段 Notebook。
